# Comparação de modelos de embeddings

Recall@k e MRR por modelo × categoria de reescrita, com o conjunto de teste do 05 (`results/teste_reescrita.csv`).

**`INDICE`:** `"2022"` = preliminar (1.852 alegações); `"ambos"` = **avaliação definitiva** (2022 + Central de Fatos, ~12 mil alegações), a rodar depois da rotulagem entre ciclos. Os acertos são definidos pelo texto da alegação, então o mesmo conjunto de teste vale nos dois índices.

**Regra de escolha:** pré-registrada no `design.md` antes da rodada definitiva.

**Consulta:** avaliada `bruta` e com a normalização `condicional` (apelidos sempre; Enelvo só quando a consulta tem cara de internetês), a melhor do 05.

Saídas: `results/comparacao_modelos_<INDICE>.csv`; embeddings do índice em cache em `data/emb/` (DVC).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from avaliacao import linhas_com_texto, metricas, posicoes
from corpora import carregar_indice
from modelos import CACHE, MODELOS, carregar, embeddings
from normalizacao import criar_enelvo, normalizar_se_preciso, vocabulario_protegido
from reescrita import carregar_apelidos, carregar_internetes

INDICE = "2022"          # "2022" (preliminar) ou "ambos" (definitiva)
RESULTS = Path("results")
CACHE.mkdir(parents=True, exist_ok=True)

indice = carregar_indice(INDICE)
teste = pd.read_csv(RESULTS / "teste_reescrita.csv").fillna({"excluir_texto": ""})
print(f"índice {INDICE}: {len(indice)} alegações ({indice['origem'].value_counts().to_dict()}) | {len(teste)} consultas")

índice 2022: 1852 alegações ({'2022': 1852}) | 534 consultas


## Consultas: bruta e normalizada

O vocabulário protegido do Enelvo vem do índice, então muda com o `INDICE`.

In [2]:
apelidos, tabela_internetes = carregar_apelidos(), carregar_internetes()
enelvo = criar_enelvo(vocabulario_protegido(indice["claim"], tabela_internetes), CACHE / f"protegidos_{INDICE}.txt")
teste["condicional"] = [normalizar_se_preciso(t, enelvo, apelidos, tabela_internetes) for t in teste["consulta"]]

relevantes = linhas_com_texto(indice["claim"], teste["alvo"])
excluir = linhas_com_texto(indice["claim"], teste["excluir_texto"])
assert all(len(r) for r in relevantes), "alvo fora do índice"

## Rodar os modelos

Para cada modelo: embeddings do índice (do cache, se já existirem para esta assinatura), embeddings das consultas, posição do alvo e métricas por categoria. O tempo de codificação entra no resultado: custo de deploy.

In [3]:
linhas = []
for familia, nome, revisao, prefixo in MODELOS:
    model = carregar(nome, revisao)
    E, seg_por_mil = embeddings(indice["claim"], nome, revisao, prefixo)   # cache em data/emb/ pelo hash dos textos
    for variante in ["consulta", "condicional"]:
        Q = model.encode([prefixo + t for t in teste[variante]], normalize_embeddings=True, batch_size=32)
        pos = posicoes(Q, E, relevantes, excluir)
        for categoria, g in teste.assign(pos=pos).groupby("categoria"):
            linhas.append({"familia": familia, "modelo": nome, "variante": "bruta" if variante == "consulta" else variante,
                           "categoria": categoria, **metricas(g["pos"].to_numpy()), "seg_por_mil_textos": seg_por_mil})
    print(f"✓ {nome}")

resultado = pd.DataFrame(linhas)
resultado.insert(0, "indice", INDICE)
resultado.to_csv(RESULTS / f"comparacao_modelos_{INDICE}.csv", index=False)

/Users/aluno2/Residencia/RES-IA-Challenge-1/.venv-1/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 58/58 [00:01<00:00, 39.69it/s]


✓ sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Batches: 100%|██████████| 58/58 [00:03<00:00, 14.77it/s]


✓ sentence-transformers/paraphrase-multilingual-mpnet-base-v2


Batches: 100%|██████████| 58/58 [00:04<00:00, 13.66it/s]


...

✓ BAAI/bge-m3


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 62578.08it/s]
[transformers] BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 58/58 [00:03<00:00, 15.32it/s]


✓ neuralmind/bert-base-portuguese-cased


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 7704.76it/s]
Invalid model-index. Not loading eval results into CardData.
Batches: 100%|██████████| 58/58 [00:13<00:00,  4.44it/s]


✓ rufimelo/bert-large-portuguese-cased-sts


## Resumo

Recall@5 por categoria, com a média entre categorias (macro: cada categoria pesa igual, independente do n).

In [4]:
for variante in ["bruta", "condicional"]:
    r = resultado[resultado["variante"] == variante]
    tabela = r.pivot(index="modelo", columns="categoria", values="recall@5")
    tabela["média"] = tabela.mean(axis=1)
    print(f"\n=== Recall@5, consulta {variante} ===")
    display(tabela.sort_values("média", ascending=False).round(2))


=== Recall@5, consulta bruta ===


categoria,apelido,apelido+girias,digitacao,girias,negacao,parafrase_real,média
modelo,,,,,,,
BAAI/bge-m3,1.0,1.00,1.00,1.00,0.99,0.88,0.98
intfloat/multilingual-e5-base,1.0,0.99,1.00,1.00,0.99,0.85,0.97
rufimelo/bert-large-portuguese-cased-sts,1.0,0.99,1.00,1.00,0.98,0.85,0.97
neuralmind/bert-base-portuguese-cased,1.0,0.98,1.00,0.99,0.99,0.79,0.96
sentence-transformers/paraphrase-multilingual-mpnet-base-v2,1.0,0.85,1.00,0.93,0.94,0.88,0.93
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2,1.0,0.80,0.99,0.89,0.99,0.88,0.93



=== Recall@5, consulta condicional ===


categoria,apelido,apelido+girias,digitacao,girias,negacao,parafrase_real,média
modelo,,,,,,,
BAAI/bge-m3,1.0,1.0,1.00,1.0,0.99,0.88,0.98
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2,1.0,1.0,0.99,1.0,0.99,0.88,0.98
intfloat/multilingual-e5-base,1.0,1.0,1.00,1.0,0.99,0.85,0.97
rufimelo/bert-large-portuguese-cased-sts,1.0,1.0,1.00,1.0,0.98,0.85,0.97
sentence-transformers/paraphrase-multilingual-mpnet-base-v2,1.0,1.0,1.00,1.0,0.94,0.88,0.97
neuralmind/bert-base-portuguese-cased,1.0,1.0,1.00,1.0,0.99,0.79,0.96
